In [4]:
from ollama import chat
import re
import fitz
from tqdm import tqdm
import os
import google.generativeai as genai
import time
import json
from dotenv import load_dotenv
import warnings

load_dotenv()
warnings.filterwarnings("ignore")

class PDFProcessor:
    def __init__(self, model_name: str = "qwen2.5:3b"):
        self.model_name = model_name
        self.SYSTEM_PROMPT = """
        You read ONE page of a financial report.

        Task:
        List the main topics discussed on this page.

        Rules:
        - Topics must be short noun phrases.
        - Only include topics clearly mentioned.
        - No explanations.

        Output:
        Return ONLY a JSON array of strings.
        """

    def extract_features_clean(self, text: str) -> list[str]:
        return list(dict.fromkeys(s.strip() for s in re.findall(r'"([^"]+)"', text)))

    def extract_page_topics(
        self,
        page_content: str,
    ) -> list[str]:
        response = chat(
            model=self.model_name,
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": f"""
                    Page content:
                    \"\"\"
                    {page_content}
                    \"\"\"
                    """,
                },
            ],
            options={
                "temperature": 0.0,
                "num_ctx": 8192,
                "num_predict": 256,
                "top_p": 0.9,
                "repeat_penalty": 1.1,
            },
        )
        raw = response.message.content.strip()
        return raw

    def read_pdf_by_page(self, path_pdf: str):
        doc = fitz.open(path_pdf)
        results_list = []
        result_text = ""

        for page_index, page in enumerate(
            tqdm(doc, desc="Reading PDF pages", unit="page"), start=1
        ):
            text = page.get_text().strip()
            ingredient = {
                "page": page_index,
                "text": self.extract_features_clean(self.extract_page_topics(text)),
            }
            results_list.append(ingredient)
            result_text += " " + str(ingredient)

        return results_list, result_text

class BuildTree:
    def __init__(self, mini_model):
        self.mini_model = mini_model
        self.pdf_processor = PDFProcessor(model_name=mini_model)

    # ==============================
    # Cache Path Builder
    # ==============================
    def get_cache_path(self, pdf_path, target_site, report_type, root_dir="tree_cache"):

        filename = os.path.basename(pdf_path)
        name_no_ext = os.path.splitext(filename)[0]

        folder = os.path.join(
            root_dir,
            target_site,
            report_type,
            name_no_ext
        )

        os.makedirs(folder, exist_ok=True)

        return os.path.join(folder, "tree.json")

    # ==============================
    # Save Tree JSON
    # ==============================
    def save_tree(self, tree_data, cache_path):
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(tree_data, f, indent=2, ensure_ascii=False)

        print(f"💾 Tree saved → {cache_path}")

    # ==============================
    # Main Run (Skip if Exists)
    # ==============================
    def run(self, file_paths, type_report, target_site):

        use_paths = file_paths[target_site][type_report]

        TREES = {}
        print("\n🚀 Start GenTree Pipeline")

        total_time = 0

        for pdf_path in use_paths:

            # ✅ Cache file path
            cache_path = self.get_cache_path(
                pdf_path,
                target_site,
                type_report
            )

            # ======================================
            # ✅ Skip if tree already exists
            # ======================================
            if os.path.exists(cache_path):
                print(f"⏭️ SKIP (tree exists): {cache_path}")
                continue

            # ======================================
            # Generate Tree (Only if missing)
            # ======================================
            print(f"🌱 Generating tree for: {pdf_path}")

            start = time.time()

            results_list, _ = self.pdf_processor.read_pdf_by_page(pdf_path)

            elapsed = time.time() - start
            total_time += elapsed

            print(f"✅ Done in {elapsed:.2f}s")

            # ✅ Save tree.json
            self.save_tree(results_list, cache_path)

            TREES[pdf_path] = results_list

        print("--" * 50)
        print("🎉 Finished GenTree")
        print("Total generation time:", round(total_time, 2), "seconds")

        return TREES

In [ ]:
# # 1
# target_site = "hd.eneos.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/hd.eneos.co.jp/00.pdf",
#         ],
#         "governance_report": ["data/hd.eneos.co.jp/system_governance_report.pdf"],
#     }
# }

# # 2
# target_site = "mitsubishicorp.com"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/mitsubishicorp.com/all.pdf",
#         ],
#         "governance_report": ["data/mitsubishicorp.com/governance_report_j.pdf"],
#     }
# }

# # 3
# target_site = "lasertec.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/lasertec.co.jp/6920_ir_material_for_fiscal_ym15_192733_00.pdf",
#         ],
#         "governance_report": ["data/lasertec.co.jp/6920_tdnet_2691142_00.pdf"],
#     }
# }

# # 4
# target_site = "itochu.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/itochu.co.jp/ja_ir_download___icsFiles_afieldfile_2025_09_05_ar2025J.pdf",
#         ],
#         "governance_report": ["data/itochu.co.jp/ja_files_corporate_governance.pdf"],
#     }
# }

# # 5
# target_site = "casio.com"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/casio.com/content_dam_casio_global_corporate_ir_library_annual_2025_integrated-2025.pdf",
#         ],
#         "governance_report": ["data/casio.com/disclosure_20251224_20251218522473.pdf"],
#     }
# }

# # 6
# target_site = "boi.jp"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/boi.jp/xcontents_AS80485_6a8dfa8d_7071_47c1_ac15_9a4be8a876b8_140120251113500915.pdf",
#             "data/boi.jp/xcontents_AS80485_2e4d59dd_a617_4b86_b05d_2fbfda9ec6a2_S100XD6Y.pdf",
#         ],
#         "governance_report": ["data/boi.jp/files_tdnet_140120251119506073.pdf"],
#     }
# }

# # 7
# target_site = "mol.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/mol.co.jp/ja_ir_library_integrated_report_main_01_teaserItems2_0_linkList_0_link__J_MOL_20REPORT_2025.pdf"
#         ],
#         "governance_report": [
#             "data/mol.co.jp/sustainability_governance_corporate_policy_pdf_governance-report.pdf"
#         ],
#     }
# }

# # 8 
# target_site = "nintendo.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": ["data/nintendo.co.jp/ir_pdf_2025_annual2503e.pdf"],
#         "governance_report": ["data/nintendo.co.jp/ir_en_management_governance.pdf"],
#     }
# }

# # 9
# target_site = "global.toyota"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/global.toyota/pages_global_toyota_ir_library_annual_2024_001_integrated_en.pdf"
#         ],
#         "governance_report": ["data/global.toyota/files_tdnet_140120250721517384.pdf"],
#     }
# }

# # 10
# target_site = "dena.com"
# file_paths = {
#     target_site: {
#         "ir_report": ["data/dena.com/00_2025_en.pdf"],
#         "governance_report": ["data/dena.com/files_tdnet_140120251112598427.pdf"],
#     }
# }

# # 11
# target_site = "lycorp.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": ["data/lycorp.co.jp/integrated_report_FY2024_jp.pdf"],
#         "governance_report": ["data/lycorp.co.jp/files_tdnet_140120251226527170.pdf"],
#     }
# }

# # 12
# target_site = "shinetsu.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": ["data/shinetsu.co.jp/統合報告書2025.pdf"],
#         "governance_report": ["data/shinetsu.co.jp/files_tdnet_140120251223524981.pdf"],
#     }
# }

# # 13
# target_site = "bridgestone.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": ["data/bridgestone.co.jp/ir2025_single.pdf"],
#         "governance_report": [
#             "data/bridgestone.co.jp/files_tdnet_140120251031583941.pdf"
#         ],
#     }
# }

# # 14
# target_site = "capcom.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/capcom.co.jp/ir_english_data_pdf_annual_2025_annual_2025_01.pdf"
#         ],
#         "governance_report": ["data/capcom.co.jp/files_tdnet_140120260106529563.pdf"],
#     }
# }

# # 15
# target_site = "fastretailing.com"
# file_paths = {
#     target_site: {
#         "ir_report": ["data/fastretailing.com/jp_ir_library_pdf_ar2024.pdf"],
#         "governance_report": [
#             "data/fastretailing.com/jp_about_governance_pdf_governance_report.pdf"
#         ],
#     }
# }

# # 16
# target_site = "group.softbank"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/group.softbank/media_Project_sbg_sbg_pdf_ir_financials_annual_reports_annual-report_fy2025_ja.pdf"
#         ],
#         "governance_report": [
#             "data/group.softbank/media_Project_sbg_sbg_pdf_about_corporate_governance_governance_20250704_01_ja.pdf"
#         ],
#     }
# }

# # 17
# target_site = "daiichisankyo.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/daiichisankyo.co.jp/files_investors_library_annual_report_index_VR2025_ds_vr2025_all_1119.pdf"
#         ],
#         "governance_report": [
#             "data/daiichisankyo.co.jp/files_tdnet_140120251218521839.pdf"
#         ],
#     }
# }

# # 18
# target_site = "kajima.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/kajima.co.jp/english_sustainability_report_2025_pdf_ir_e_all_2.pdf"
#         ],
#         "governance_report": ["data/kajima.co.jp/files_tdnet_140120250612588145.pdf"],
#     }
# }

# # 19
# target_site = "mhi.com"
# file_paths = {
#     target_site: {
#         "ir_report": ["data/mhi.com/jp_finance_library_annual_pdf_report_2025.pdf"],
#         "governance_report": ["data/mhi.com/files_tdnet_140120250624597698.pdf"],
#     }
# }

# # 20
# target_site = "mitsuifudosan.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/mitsuifudosan.co.jp/corporate_ir_library_integratedreport_pdf_IR2025_ja.pdf"
#         ],
#         "governance_report": [
#             "data/mitsuifudosan.co.jp/files_tdnet_140120250514552438.pdf"
#         ],
#     }
# }

# # 21
# target_site = "koeitecmo.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": ["data/koeitecmo.co.jp/files_tdnet_140120251104586205.pdf"],
#         "governance_report": [
#             "data/koeitecmo.co.jp/files_tdnet_140120250529573336.pdf"
#         ],
#     }
# }

# # 22
# target_site = "toei-anim.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/toei-anim.co.jp/en_ir_library_Report_main_00_teaserItems1_0_linkList_0_link_PEROS_20REPORT_202024_en_open.pdf"
#         ],
#         "governance_report": [
#             "data/toei-anim.co.jp/files_tdnet_140120250604582039.pdf"
#         ],
#     }
# }

# # 23
# target_site = "mufg.jp"
# file_paths = {
#     target_site: {
#         "ir_report": ["data/mufg.jp/dam_ir_presentation_2025_pdf_slides2509_ja.pdf"],
#         "governance_report": ["data/mufg.jp/files_tdnet_140120251107591892.pdf"],
#     }
# }

# # 24
# target_site = "jfe-holdings.co.jp"
# file_paths = {
#     target_site: {
#         "ir_report": [
#             "data/jfe-holdings.co.jp/common_pdf_investor_library_group-report_2025_all_A4.pdf"
#         ],
#         "governance_report": [
#             "data/jfe-holdings.co.jp/en_common_pdf_company_info_corporate-governance.pdf"
#         ],
#     }
# }

# # 25
# target_site = "advantest.com"
# file_paths = {
#     target_site: {
#         "ir_report": ["data/advantest.com/E_all_IAR2025.pdf"],
#         "governance_report": ["data/advantest.com/files_tdnet_140120251126509485.pdf"],
#     }
# }

In [5]:
file_paths = {
    "hd.eneos.co.jp": {
        "ir_report": [
            "data/hd.eneos.co.jp/00.pdf",
        ],
        "governance_report": [
            "data/hd.eneos.co.jp/system_governance_report.pdf"
        ],
    },

    "mitsubishicorp.com": {
        "ir_report": [
            "data/mitsubishicorp.com/all.pdf",
        ],
        "governance_report": [
            "data/mitsubishicorp.com/governance_report_j.pdf"
        ],
    },

    "lasertec.co.jp": {
        "ir_report": [
            "data/lasertec.co.jp/6920_ir_material_for_fiscal_ym15_192733_00.pdf",
        ],
        "governance_report": [
            "data/lasertec.co.jp/6920_tdnet_2691142_00.pdf"
        ],
    },

    "itochu.co.jp": {
        "ir_report": [
            "data/itochu.co.jp/ja_ir_download___icsFiles_afieldfile_2025_09_05_ar2025J.pdf",
        ],
        "governance_report": [
            "data/itochu.co.jp/ja_files_corporate_governance.pdf"
        ],
    },

    "casio.com": {
        "ir_report": [
            "data/casio.com/content_dam_casio_global_corporate_ir_library_annual_2025_integrated-2025.pdf",
        ],
        "governance_report": [
            "data/casio.com/disclosure_20251224_20251218522473.pdf"
        ],
    },

    "boi.jp": {
        "ir_report": [
            "data/boi.jp/xcontents_AS80485_6a8dfa8d_7071_47c1_ac15_9a4be8a876b8_140120251113500915.pdf",
            "data/boi.jp/xcontents_AS80485_2e4d59dd_a617_4b86_b05d_2fbfda9ec6a2_S100XD6Y.pdf",
        ],
        "governance_report": [
            "data/boi.jp/files_tdnet_140120251119506073.pdf"
        ],
    },

    "mol.co.jp": {
        "ir_report": [
            "data/mol.co.jp/ja_ir_library_integrated_report_main_01_teaserItems2_0_linkList_0_link__J_MOL_20REPORT_2025.pdf"
        ],
        "governance_report": [
            "data/mol.co.jp/sustainability_governance_corporate_policy_pdf_governance-report.pdf"
        ],
    },

    "nintendo.co.jp": {
        "ir_report": [
            "data/nintendo.co.jp/ir_pdf_2025_annual2503e.pdf"
        ],
        "governance_report": [
            "data/nintendo.co.jp/ir_en_management_governance.pdf"
        ],
    },

    "global.toyota": {
        "ir_report": [
            "data/global.toyota/pages_global_toyota_ir_library_annual_2024_001_integrated_en.pdf"
        ],
        "governance_report": [
            "data/global.toyota/files_tdnet_140120250721517384.pdf"
        ],
    },

    "dena.com": {
        "ir_report": [
            "data/dena.com/00_2025_en.pdf"
        ],
        "governance_report": [
            "data/dena.com/files_tdnet_140120251112598427.pdf"
        ],
    },

    "lycorp.co.jp": {
        "ir_report": [
            "data/lycorp.co.jp/integrated_report_FY2024_jp.pdf"
        ],
        "governance_report": [
            "data/lycorp.co.jp/files_tdnet_140120251226527170.pdf"
        ],
    },

    "shinetsu.co.jp": {
        "ir_report": [
            "data/shinetsu.co.jp/統合報告書2025.pdf"
        ],
        "governance_report": [
            "data/shinetsu.co.jp/files_tdnet_140120251223524981.pdf"
        ],
    },

    "bridgestone.co.jp": {
        "ir_report": [
            "data/bridgestone.co.jp/ir2025_single.pdf"
        ],
        "governance_report": [
            "data/bridgestone.co.jp/files_tdnet_140120251031583941.pdf"
        ],
    },

    "capcom.co.jp": {
        "ir_report": [
            "data/capcom.co.jp/ir_english_data_pdf_annual_2025_annual_2025_01.pdf"
        ],
        "governance_report": [
            "data/capcom.co.jp/files_tdnet_140120260106529563.pdf"
        ],
    },

    "fastretailing.com": {
        "ir_report": [
            "data/fastretailing.com/jp_ir_library_pdf_ar2024.pdf"
        ],
        "governance_report": [
            "data/fastretailing.com/jp_about_governance_pdf_governance_report.pdf"
        ],
    },

    "group.softbank": {
        "ir_report": [
            "data/group.softbank/media_Project_sbg_sbg_pdf_ir_financials_annual_reports_annual-report_fy2025_ja.pdf"
        ],
        "governance_report": [
            "data/group.softbank/media_Project_sbg_sbg_pdf_about_corporate_governance_governance_20250704_01_ja.pdf"
        ],
    },

    "daiichisankyo.co.jp": {
        "ir_report": [
            "data/daiichisankyo.co.jp/files_investors_library_annual_report_index_VR2025_ds_vr2025_all_1119.pdf"
        ],
        "governance_report": [
            "data/daiichisankyo.co.jp/files_tdnet_140120251218521839.pdf"
        ],
    },

    "kajima.co.jp": {
        "ir_report": [
            "data/kajima.co.jp/english_sustainability_report_2025_pdf_ir_e_all_2.pdf"
        ],
        "governance_report": [
            "data/kajima.co.jp/files_tdnet_140120250612588145.pdf"
        ],
    },

    "mhi.com": {
        "ir_report": [
            "data/mhi.com/jp_finance_library_annual_pdf_report_2025.pdf"
        ],
        "governance_report": [
            "data/mhi.com/files_tdnet_140120250624597698.pdf"
        ],
    },

    "mitsuifudosan.co.jp": {
        "ir_report": [
            "data/mitsuifudosan.co.jp/corporate_ir_library_integratedreport_pdf_IR2025_ja.pdf"
        ],
        "governance_report": [
            "data/mitsuifudosan.co.jp/files_tdnet_140120250514552438.pdf"
        ],
    },

    "koeitecmo.co.jp": {
        "ir_report": [
            "data/koeitecmo.co.jp/files_tdnet_140120251104586205.pdf"
        ],
        "governance_report": [
            "data/koeitecmo.co.jp/files_tdnet_140120250529573336.pdf"
        ],
    },

    "toei-anim.co.jp": {
        "ir_report": [
            "data/toei-anim.co.jp/en_ir_library_Report_main_00_teaserItems1_0_linkList_0_link_PEROS_20REPORT_202024_en_open.pdf"
        ],
        "governance_report": [
            "data/toei-anim.co.jp/files_tdnet_140120250604582039.pdf"
        ],
    },

    "mufg.jp": {
        "ir_report": [
            "data/mufg.jp/dam_ir_presentation_2025_pdf_slides2509_ja.pdf"
        ],
        "governance_report": [
            "data/mufg.jp/files_tdnet_140120251107591892.pdf"
        ],
    },

    "jfe-holdings.co.jp": {
        "ir_report": [
            "data/jfe-holdings.co.jp/common_pdf_investor_library_group-report_2025_all_A4.pdf"
        ],
        "governance_report": [
            "data/jfe-holdings.co.jp/en_common_pdf_company_info_corporate-governance.pdf"
        ],
    },

    "advantest.com": {
        "ir_report": [
            "data/advantest.com/E_all_IAR2025.pdf"
        ],
        "governance_report": [
            "data/advantest.com/files_tdnet_140120251126509485.pdf"
        ],
    },
}

In [ ]:
builder = BuildTree(mini_model="qwen2.5:3b")

ALL_TREES = {}

for target_site in file_paths:

    print("\n" + "=" * 80)
    print(f"🌍 Processing site: {target_site}")
    print("=" * 80)

    ALL_TREES[target_site] = {}

    # Loop từng loại report
    for report_type in file_paths[target_site]:

        print("\n" + "-" * 60)
        print(f"📄 Report type: {report_type}")
        print("-" * 60)

        TREES = builder.run(
            file_paths=file_paths,
            type_report=report_type,
            target_site=target_site
        )

        ALL_TREES[target_site][report_type] = TREES

print("\n🎉 Finished processing ALL sites!")


🌍 Processing site: hd.eneos.co.jp

------------------------------------------------------------
📄 Report type: ir_report
------------------------------------------------------------

🚀 Start GenTree Pipeline
🌱 Generating tree for: data/hd.eneos.co.jp/00.pdf


Reading PDF pages:  40%|███▉      | 35/88 [02:31<03:37,  4.11s/page]